In [1]:
import sys, torch
print("python:", sys.version.split()[0])   # expect 3.14.x
print("torch :", torch.__version__)         # expect 2.13.0+cpu

python: 3.14.5
torch : 2.13.0+cpu


LSTM Step 1 — Build the sequences

In [2]:
# LSTM Step 1: build 168-hour sequences from a subset of buildings
import numpy as np, pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
np.random.seed(42)

df = pd.read_parquet("model_ready_clean.parquet")

# ---- pick a subset (CPU-feasible) ----
N_BUILD = 40
L = 168                                   # window length = 1 week
TRAIN_END = "2016-09-30 23:00:00"
subset_b = np.random.choice(df["building_id"].unique(), size=N_BUILD, replace=False)
sub = df[df["building_id"].isin(subset_b)].copy()

# ---- scale air_temperature using TRAIN period only (no leakage) ----
tr = sub[sub["timestamp"] <= TRAIN_END]
t_mean, t_std = tr["air_temperature"].mean(), tr["air_temperature"].std()

# ---- build per-building windows: features = [log_reading, scaled_temp] ----
X_list, y_list, meta_list = [], [], []
for bid, g in sub.groupby("building_id"):
    g = g.sort_values("timestamp")
    if len(g) <= L:
        continue
    logr = np.log1p(g["meter_reading"].values).astype("float32")
    temp = ((g["air_temperature"].values - t_mean) / t_std).astype("float32")
    feats = np.stack([logr, temp], axis=1)                     # (T, 2)
    win = sliding_window_view(feats, L, axis=0).transpose(0, 2, 1)[:-1]  # (T-L, L, 2)
    X_list.append(win)
    y_list.append(logr[L:])                                    # next-hour log reading
    meta_list.append(pd.DataFrame({"building_id": bid,
                                    "timestamp": g["timestamp"].values[L:],
                                    "y_true_kwh": g["meter_reading"].values[L:]}))

X = np.concatenate(X_list).astype("float32")
y = np.concatenate(y_list).astype("float32")
meta = pd.concat(meta_list).reset_index(drop=True)

# ---- time-based split (same Sep30 cutoff) ----
is_train = (meta["timestamp"] <= TRAIN_END).values
Xtr, ytr = X[is_train], y[is_train]
Xte, yte = X[~is_train], y[~is_train]

print("all sequences:", X.shape, "| features per step:", X.shape[2])
print("train:", Xtr.shape, "test:", Xte.shape)


all sequences: (279076, 168, 2) | features per step: 2
train: (201542, 168, 2) test: (77534, 168, 2)


LSTM Step 2 — Define and train the network

In [3]:
# LSTM Step 2: define + train the LSTM
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42)

train_dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                      batch_size=512, shuffle=True)

class LSTMRegressor(nn.Module):
    def __init__(self, n_feat=2, hidden=48):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, batch_first=True)  # reads the 168-hr sequence
        self.head = nn.Linear(hidden, 1)                       # -> next-hour prediction
    def forward(self, x):
        _, (h, _) = self.lstm(x)          # h[-1] = final hidden state (summary of the week)
        return self.head(h[-1]).squeeze(-1)

model  = LSTMRegressor()
opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 3
for ep in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for xb, yb in train_dl:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        running += loss.item() * len(xb)
    print(f"epoch {ep}: train MSE (log) = {running/len(Xtr):.4f}")
print("✅ training done")


epoch 1: train MSE (log) = 2.1115
epoch 2: train MSE (log) = 0.0637
epoch 3: train MSE (log) = 0.0535
✅ training done


LSTM Step 3a — Evaluate the LSTM on the test set

In [4]:
# LSTM Step 3a: predict + LSTM metrics
model.eval()
with torch.no_grad():
    lstm_log = model(torch.from_numpy(Xte)).numpy()
lstm_kwh = np.clip(np.expm1(lstm_log), 0, None)   # back to kWh

test_meta = meta[~is_train].copy()
test_meta["lstm_pred"] = lstm_kwh

# metrics (redefined here since this is a fresh notebook)
def rmse(y, p):  return np.sqrt(np.mean((y - p) ** 2))
def mae(y, p):   return np.mean(np.abs(y - p))
def rmsle(y, p): p = np.clip(p, 0, None); return np.sqrt(np.mean((np.log1p(y) - np.log1p(p))**2))
def mape(y, p):  m = y > 0; return np.mean(np.abs((y[m] - p[m]) / y[m])) * 100

yv = test_meta["y_true_kwh"].values
print("LSTM on subset test set:")
print(f"  RMSE  : {rmse(yv, lstm_kwh):.3f}")
print(f"  MAE   : {mae(yv, lstm_kwh):.3f}")
print(f"  MAPE% : {mape(yv, lstm_kwh):.3f}")
print(f"  RMSLE : {rmsle(yv, lstm_kwh):.3f}")


LSTM on subset test set:
  RMSE  : 153.896
  MAE   : 35.130
  MAPE% : 11.685
  RMSLE : 0.199


LSTM Step 3b — Fair 3-way comparison

In [5]:
# LSTM Step 3b: fair 3-way comparison (same subset, same test points)
import lightgbm as lgb

features = ["hour","weekday","month","is_weekend","lag_24h","lag_168h","roll_mean_24h",
            "air_temperature","dew_temperature","log_square_feet","primary_use","building_id"]
cat_features = ["primary_use", "building_id"]

sub_feat = df[df["building_id"].isin(subset_b)].copy()
sub_feat["primary_use"] = sub_feat["primary_use"].astype("category")
sub_feat["building_id"] = sub_feat["building_id"].astype("int").astype("category")

strain = sub_feat[sub_feat["timestamp"] <= TRAIN_END]
stest  = sub_feat[sub_feat["timestamp"] >= "2016-10-01 00:00:00"].copy()

lgbm = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=63,
                         subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
lgbm.fit(strain[features], np.log1p(strain["meter_reading"]), categorical_feature=cat_features)

stest["lgbm_pred"]  = np.clip(np.expm1(lgbm.predict(stest[features])), 0, None)
stest["naive_pred"] = stest["lag_168h"].values
stest["building_id"] = stest["building_id"].astype(int)

# align all three models on identical (building_id, timestamp) points
tm = test_meta.copy(); tm["building_id"] = tm["building_id"].astype(int)
comp = tm.merge(stest[["building_id", "timestamp", "lgbm_pred", "naive_pred"]],
                on=["building_id", "timestamp"], how="inner")

yv = comp["y_true_kwh"].values
table = {}
for name, col in [("Seasonal-naive", "naive_pred"), ("LightGBM", "lgbm_pred"), ("LSTM", "lstm_pred")]:
    p = comp[col].values
    table[name] = {"RMSE": rmse(yv, p), "MAE": mae(yv, p), "MAPE%": mape(yv, p), "RMSLE": rmsle(yv, p)}

print("Fair 3-way comparison (same 40 buildings, same test points):")
print(pd.DataFrame(table).T[["RMSE", "MAE", "MAPE%", "RMSLE"]].round(3))
print("\ncommon test points:", len(comp))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002686 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1317
[LightGBM] [Info] Number of data points in the train set: 207966, number of used features: 12
[LightGBM] [Info] Start training from score 4.392755
Fair 3-way comparison (same 40 buildings, same test points):
                   RMSE     MAE   MAPE%  RMSLE
Seasonal-naive   90.954  23.802  14.189  0.293
LightGBM         70.095  19.706  11.060  0.218
LSTM            153.896  35.130  11.685  0.199

common test points: 77534
